# 1. Gather GeLU Input/Output activation for evaluation

In [1]:
import pandas as pd
from math import ceil

# -----------------------------
# Energy coefficients (example)
# -----------------------------
ENERGY = {
    'mac_pj': 0.2,              # pJ per MAC
    'dram_pj_per_byte': 200.0,  # pJ per off-chip byte
    'sram_pj_per_byte': 20.0,   # pJ per on-chip SRAM byte
}

# --------------------------------
# Systolic array / tiling settings
# --------------------------------
CFG = {
    'Tm': 32,
    'Tn': 32,
    'Tk': 64,
    'data_bytes': 1,   # INT8 activations / weights
    'psum_bytes': 4,   # 32-bit partial sums
}

# ---------------------------------------
# Example ViT-B/16-like operator settings
# ---------------------------------------
layers = [
    {'name': 'qkv_proj',   'M': 197, 'K': 768,  'N': 2304, 'heads': 1,  'transpose_extra': 0},
    {'name': 'attn_score', 'M': 197, 'K': 64,   'N': 197,  'heads': 12, 'transpose_extra': 197 * 64},
    {'name': 'attn_value', 'M': 197, 'K': 197,  'N': 64,   'heads': 12, 'transpose_extra': 0},
    {'name': 'out_proj',   'M': 197, 'K': 768,  'N': 768,  'heads': 1,  'transpose_extra': 0},
    {'name': 'mlp_fc1',    'M': 197, 'K': 768,  'N': 3072, 'heads': 1,  'transpose_extra': 0},
    {'name': 'mlp_fc2',    'M': 197, 'K': 3072, 'N': 768,  'heads': 1,  'transpose_extra': 0},
]

def estimate_layer(layer, dataflow='OS'):
    M, K, N, H = layer['M'], layer['K'], layer['N'], layer['heads']
    Tm, Tn, Tk = CFG['Tm'], CFG['Tn'], CFG['Tk']
    b, pb = CFG['data_bytes'], CFG['psum_bytes']

    mt = ceil(M / Tm)
    nt = ceil(N / Tn)
    kt = ceil(K / Tk)

    macs = M * K * N * H

    # Base stream traffic for tiled GEMM (same backbone for all dataflows)
    base_stream_bytes = H * (mt * nt * kt * (Tm * Tk + Tk * Tn) * b + M * N * b)

    # Extra traffic caused by dataflow choice
    if dataflow == 'OS':
        psum_spill_bytes = 0
        transpose_bytes = 0
    elif dataflow in ('WS_spill', 'IS_spill'):
        # A proxy model: if partial sums are spilled between K tiles
        psum_spill_bytes = H * (max(kt - 1, 0) * mt * nt * Tm * Tn * 2 * pb)
        # Explicit K transpose / reordering overhead for attention-score path
        transpose_bytes = H * layer['transpose_extra'] * b
    else:
        raise ValueError(f'Unknown dataflow: {dataflow}')

    dram_bytes = base_stream_bytes + transpose_bytes
    sram_bytes = psum_spill_bytes

    total_energy_pj = (
        macs * ENERGY['mac_pj']
        + dram_bytes * ENERGY['dram_pj_per_byte']
        + sram_bytes * ENERGY['sram_pj_per_byte']
    )

    return {
        'layer': layer['name'],
        'dataflow': dataflow,
        'macs': macs,
        'dram_bytes': dram_bytes,
        'sram_psum_bytes': sram_bytes,
        'transpose_bytes': transpose_bytes,
        'total_energy_uJ': total_energy_pj / 1e6,
    }

def run_all():
    rows = []
    for layer in layers:
        for df in ['OS', 'WS_spill', 'IS_spill']:
            rows.append(estimate_layer(layer, df))

    df = pd.DataFrame(rows)

    summary = (
        df.pivot_table(
            index='dataflow',
            values=['total_energy_uJ', 'dram_bytes', 'sram_psum_bytes'],
            aggfunc='sum'
        )
        .reset_index()
    )

    os_energy = summary.loc[summary['dataflow'] == 'OS', 'total_energy_uJ'].iloc[0]
    summary['energy_vs_OS'] = summary['total_energy_uJ'] / os_energy

    return df, summary

if __name__ == '__main__':
    layerwise, summary = run_all()
    layerwise.to_csv('vit_os_energy_layerwise.csv', index=False)
    summary.to_csv('vit_os_energy_summary.csv', index=False)

    print('=== Layer-wise ===')
    print(layerwise)
    print('\n=== Summary ===')
    print(summary)


=== Layer-wise ===
         layer  dataflow       macs  dram_bytes  sram_psum_bytes  \
0     qkv_proj        OS  348585984    25226496                0   
1     qkv_proj  WS_spill  348585984    25226496         45416448   
2     qkv_proj  IS_spill  348585984    25226496         45416448   
3   attn_score        OS   29805312     2874156                0   
4   attn_score  WS_spill   29805312     3025452                0   
5   attn_score  IS_spill   29805312     3025452                0   
6   attn_value        OS   29805312     2903808                0   
7   attn_value  WS_spill   29805312     2903808          4128768   
8   attn_value  IS_spill   29805312     2903808          4128768   
9     out_proj        OS  116195328     8408832                0   
10    out_proj  WS_spill  116195328     8408832         15138816   
11    out_proj  IS_spill  116195328     8408832         15138816   
12     mlp_fc1        OS  464781312    33635328                0   
13     mlp_fc1  WS_spill  464

In [2]:
import math
import itertools
from dataclasses import dataclass
from typing import List, Tuple, Dict

# ---------------------------------------------------------
# 1. Hardware & Energy Cost Definitions
# ---------------------------------------------------------
@dataclass
class HardwareConstraints:
    # Target Xilinx FPGA Platform (e.g., similar to ZCU102 / KU115)
    max_dsp: int = 2048          # Total DSP slices available for compute
    max_bram_bytes: int = 1024 * 1024 * 10  # 10 MB On-chip BRAM
    data_width_bytes: int = 1    # INT8 precision (1 byte)

@dataclass
class EnergyModel:
    # Relative energy cost per operation (normalized to MAC)
    mac_energy: float = 1.0      # Energy per Multiply-Accumulate
    sram_read: float = 5.0       # Energy per On-chip SRAM read
    sram_write: float = 5.0      # Energy per On-chip SRAM write
    dram_read: float = 200.0     # Energy per Off-chip DRAM read
    dram_write: float = 200.0    # Energy per Off-chip DRAM write

# ---------------------------------------------------------
# 2. ViT Workload Definitions
# ---------------------------------------------------------
@dataclass
class ViTLayer:
    name: str
    M: int # Sequence length (Tokens)
    K: int # Input Embedding / Channel dimension
    N: int # Output Channel dimension
    
def get_vit_workload() -> List[ViTLayer]:
    # M = 197 (14x14 patches + 1 cls token)
    # C = 384 (Embedding dim)
    M, C, C_mlp = 197, 384, 1536
    return [
        ViTLayer("QKV_Proj", M, C, C * 3), # Q, K, V combined
        ViTLayer("Proj_Out", M, C, C),     # Output Projection
        ViTLayer("MLP_1",    M, C, C_mlp), # MLP expansion (x4)
        ViTLayer("MLP_2",    M, C_mlp, C)  # MLP reduction
    ]

# ---------------------------------------------------------
# 3. DSE Evaluation Function
# ---------------------------------------------------------
def evaluate_design_point(
    layer: ViTLayer, 
    Tm: int, Tk: int, Tn: int, 
    dataflow: str, 
    hw: HardwareConstraints, 
    energy_mod: EnergyModel
):
    """
    Evaluate a specific tiling strategy and dataflow for a given layer.
    Tm, Tk, Tn: Tile sizes for M, K, N dimensions.
    """
    # 1. Padding overhead calculation (197 is prime, requires padding)
    pad_M = math.ceil(layer.M / Tm) * Tm
    pad_K = math.ceil(layer.K / Tk) * Tk
    pad_N = math.ceil(layer.N / Tn) * Tn
    
    # 2. Map tiles to Systolic Array (SA) dimensions based on Dataflow
    # To maximize stationary data reuse, we map spatial dimensions to the SA
    if dataflow == "WS":
        # Weight Stationary: SA holds Weights. Spatial dims = Tk, Tn. Temporal = Tm.
        SA_rows, SA_cols = Tk, Tn
        temporal_steps_per_tile = Tm
    elif dataflow == "IS":
        # Input Stationary: SA holds Inputs. Spatial dims = Tm, Tk. Temporal = Tn.
        SA_rows, SA_cols = Tm, Tk
        temporal_steps_per_tile = Tn
    elif dataflow == "OS":
        # Output Stationary: SA holds Outputs. Spatial dims = Tm, Tn. Temporal = Tk.
        SA_rows, SA_cols = Tm, Tn
        temporal_steps_per_tile = Tk
    else:
        raise ValueError("Invalid Dataflow")

    # Resource check: DSPs
    if SA_rows * SA_cols > hw.max_dsp:
        return None # Invalid design, exceeds DSP limits

    # 3. Memory Footprint (Double Buffering for latency hiding)
    # Buffers needed for Act (M*K), Weight (K*N), Output (M*N)
    sram_act = 2 * (Tm * Tk) * hw.data_width_bytes
    sram_wt  = 2 * (Tk * Tn) * hw.data_width_bytes
    sram_out = 2 * (Tm * Tn) * hw.data_width_bytes
    total_sram = sram_act + sram_wt + sram_out
    
    if total_sram > hw.max_bram_bytes:
        return None # Invalid design, exceeds BRAM limits

    # 4. Data Movement Cost Estimation (Assuming optimal loop ordering for the dataflow)
    num_tiles_M = pad_M // Tm
    num_tiles_K = pad_K // Tk
    num_tiles_N = pad_N // Tn
    
    # Base ideal memory accesses if data fits on-chip
    total_compute_ops = pad_M * pad_K * pad_N
    
    # Calculate DRAM accesses based on stationary tensor
    if dataflow == "WS":
        # Weights are stationary: Load W once per K-N tile. A and O stream through M.
        dram_reads_wt  = pad_K * pad_N
        dram_reads_act = pad_M * pad_K * num_tiles_N  # Act is reread for every N tile
        dram_writes_out = pad_M * pad_N * num_tiles_K # Partial sums round-trip or accumulated in SRAM?
        # Assuming SRAM Out buffer is large enough to avoid partial sum DRAM roundtrip:
        dram_writes_out = pad_M * pad_N 
        dram_reads_out  = pad_M * pad_N * (num_tiles_K - 1) # Read partial sums if K needs multiple passes
        
    elif dataflow == "OS":
        # Outputs are stationary: O stays in SRAM/PE. W and A stream.
        dram_writes_out = pad_M * pad_N
        dram_reads_out  = 0 # Accumulated locally
        dram_reads_act  = pad_M * pad_K * num_tiles_N
        dram_reads_wt   = pad_K * pad_N * num_tiles_M
        
    elif dataflow == "IS":
        # Inputs are stationary: A stays in SRAM/PE. W and O stream.
        dram_reads_act  = pad_M * pad_K
        dram_reads_wt   = pad_K * pad_N * num_tiles_M
        dram_writes_out = pad_M * pad_N
        dram_reads_out  = pad_M * pad_N * (num_tiles_K - 1)

    # Calculate Energy
    energy_mac = total_compute_ops * energy_mod.mac_energy
    
    # SRAM accesses (Every MAC requires reading 1 Act, 1 Wt, and R/W 1 Out)
    sram_access_energy = (total_compute_ops * 2 * energy_mod.sram_read + 
                          total_compute_ops * energy_mod.sram_write)
    
    dram_access_energy = ( (dram_reads_wt + dram_reads_act + dram_reads_out) * energy_mod.dram_read +
                           dram_writes_out * energy_mod.dram_write )
                           
    total_energy = energy_mac + sram_access_energy + dram_access_energy
    
    # 5. Latency Estimation (Compute bound + Pipeline filling)
    # Simplification: Array takes time to fill/drain
    pipeline_delay = SA_rows + SA_cols
    cycles_per_tile = temporal_steps_per_tile + pipeline_delay
    total_cycles = cycles_per_tile * (num_tiles_M * num_tiles_K * num_tiles_N)

    return {
        "energy": total_energy,
        "cycles": total_cycles,
        "dram_accesses": dram_reads_wt + dram_reads_act + dram_reads_out + dram_writes_out,
        "SA_shape": (SA_rows, SA_cols),
        "utilization": (layer.M * layer.K * layer.N) / total_compute_ops # Inverse of padding waste
    }

# ---------------------------------------------------------
# 4. Main Design Space Exploration (DSE) Loop
# ---------------------------------------------------------
def run_dse():
    hw = HardwareConstraints()
    energy_mod = EnergyModel()
    layers = get_vit_workload()
    
    dataflows = ["WS", "OS", "IS"]
    
    # Define search space for Tiling (Multiples of common memory alignments)
    # Tm covers up to 197. Tk and Tn cover up to 1536.
    Tm_cands = [16, 32, 50, 66, 100, 200]  # 200 is good for padding 197 minimally
    Tk_cands = [16, 32, 64, 128, 192, 384]
    Tn_cands = [16, 32, 64, 128, 192, 384]

    best_overall_energy = float('inf')
    best_config = {}

    print("Starting Design Space Exploration for ViT on FPGA...")
    
    # Iterate through all combinations
    for df in dataflows:
        for Tm, Tk, Tn in itertools.product(Tm_cands, Tk_cands, Tn_cands):
            
            total_network_energy = 0
            total_network_cycles = 0
            is_valid = True
            
            # Evaluate this hardware configuration across all ViT layers
            for layer in layers:
                res = evaluate_design_point(layer, Tm, Tk, Tn, df, hw, energy_mod)
                if res is None:
                    is_valid = False
                    break
                
                total_network_energy += res["energy"]
                total_network_cycles += res["cycles"]
            
            if is_valid:
                # Update optimal configuration based on Energy
                if total_network_energy < best_overall_energy:
                    best_overall_energy = total_network_energy
                    best_config = {
                        "Dataflow": df,
                        "Tile_M": Tm,
                        "Tile_K": Tk,
                        "Tile_N": Tn,
                        "Energy": total_network_energy,
                        "Cycles": total_network_cycles
                    }

    # ---------------------------------------------------------
    # 5. Print Results
    # ---------------------------------------------------------
    print("\n" + "="*50)
    print("Optimal Design Configuration Found:")
    print("="*50)
    print(f"Optimal Dataflow:  {best_config['Dataflow']}")
    print(f"Matrix Tiling:     Tm={best_config['Tile_M']}, Tk={best_config['Tile_K']}, Tn={best_config['Tile_N']}")
    print(f"Total Energy Cost: {best_config['Energy']:.2e} (Arbitrary Units)")
    print(f"Total Compute Cyc: {best_config['Cycles']:.2e} Cycles")
    
    # Detailed Layer Breakdown for the best config
    print("\nLayer-by-Layer Breakdown for Best Config:")
    print("-" * 75)
    print(f"{'Layer':<10} | {'Shape(M,K,N)':<15} | {'Pad Waste':<10} | {'SA Shape(RxC)':<15} | {'DRAM Accesses'}")
    print("-" * 75)
    
    for layer in layers:
        res = evaluate_design_point(
            layer, 
            best_config["Tile_M"], best_config["Tile_K"], best_config["Tile_N"], 
            best_config["Dataflow"], hw, energy_mod
        )
        shape_str = f"({layer.M},{layer.K},{layer.N})"
        waste_str = f"{(1 - res['utilization'])*100:.1f}%"
        sa_str = f"{res['SA_shape'][0]}x{res['SA_shape'][1]}"
        print(f"{layer.name:<10} | {shape_str:<15} | {waste_str:<10} | {sa_str:<15} | {res['dram_accesses']:.2e}")

if __name__ == "__main__":
    run_dse()

Starting Design Space Exploration for ViT on FPGA...

Optimal Design Configuration Found:
Optimal Dataflow:  WS
Matrix Tiling:     Tm=66, Tk=32, Tn=64
Total Energy Cost: 9.24e+09 (Arbitrary Units)
Total Compute Cyc: 4.20e+05 Cycles

Layer-by-Layer Breakdown for Best Config:
---------------------------------------------------------------------------
Layer      | Shape(M,K,N)    | Pad Waste  | SA Shape(RxC)   | DRAM Accesses
---------------------------------------------------------------------------
QKV_Proj   | (197,384,1152)  | 0.5%       | 32x64           | 4.55e+06
Proj_Out   | (197,384,384)   | 0.5%       | 32x64           | 1.52e+06
MLP_1      | (197,384,1536)  | 0.5%       | 32x64           | 6.06e+06
MLP_2      | (197,1536,384)  | 0.5%       | 32x64           | 6.06e+06
